# EOS26 small-box test: reionization history

Runs a small box with the EOS26 template (continuous f$_{\rm esc}$) and plots its mean neutral fraction
together with the EOS26_earlyEoR production history and the Qin+25 points.

* Needs 21cmFAST ≥ 4.3 (4.2 and earlier apply the photoheating feedback to the ACG turnover mass in almost every cell; fixed in 4.3.0).
* The default box is 86³ at 1.5 cMpc (129 cMpc), seed 42, `N_THREADS = 1`, which is the box of the 14 Nov 2026 calibration plot.
  It took about 20 min on one core in the test environment (15 min with the ICs, perturbed fields and halo catalogues already cached).
  The whole simulation is written to `CACHE_DIR` (ICs, perturbed fields, halo catalogues and all coevals, about 8 GB for 86³),
  so `plot_all_summaries.ipynb` can use it when the HII_DIM = 200 test is not there. The ICs, perturbed fields and halo
  catalogues do not depend on f$_{\rm esc}$, so further runs in the same box only redo the coevals.
* Every run goes down to the lowest node redshift (z = 5), so all halo catalogues come from one merger tree rooted there, as in production.
  Do not point `CACHE_DIR` at a cache filled by a run that stopped at a higher redshift: 21cmFAST would reuse those catalogues,
  which belong to a different tree, and the history gets a step where the two trees meet. If a run is interrupted while the halo
  catalogues are being made, delete `CACHE_DIR` before running again.
* The EOS26_earlyEoR production history is read from `EOS26_earlyEoR_xHI.txt` (made from its coeval logs; rebuilt from `logs/EOS26_earlyEoR/` if the file is missing).

## Settings

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()  # postprocess/
REPO = HERE.parent
sys.path.insert(0, str(REPO / "run_scripts"))
import settings  # noqa: E402  (DATA_ROOT: $EOS26_DATA if set, /g/data/qp00/$USER on Gadi, else the repository root)

TEMPLATE = REPO / "EOS26.toml"

# Small test box, with the cell size of EOS26 (HII_DIM = 1500 at 1.5 cMpc).
HII_DIM = 86
LOWRES_CELL_SIZE_MPC = 1.5
N_THREADS = 1  # part of the RNG seeding of the ICs and halos: another value gives another realisation of the same box
RANDOM_SEED = 42

# Runs to plot. "fesc" overrides the f_esc parameters of the template ({} keeps the EOS26 values).
# F_ESC7_MCG = F_ESC10_ACG - 3 * ALPHA_ESC keeps f_esc continuous between MCGs and ACGs.
# label=None builds the label from the f_esc values.
RUNS = [
    dict(label=None, fesc={}, color="C0"),
    # dict(label=None, fesc=dict(F_ESC10_ACG=-0.85, ALPHA_ESC=-0.16, F_ESC7_MCG=-0.85), color="C1"),
]
RERUN = False  # False: use the saved history of a run with the same inputs, if there is one

CACHE_DIR = settings.DATA_ROOT / f"EOS26_smallbox_dim{HII_DIM}_cell{LOWRES_CELL_SIZE_MPC}_cache"
PROD_XHI = HERE / "EOS26_earlyEoR_xHI.txt"
PROD_LOGS = REPO / "logs" / "EOS26_earlyEoR" / "EOS26_coeval_*.log"
PLOTS = REPO / "plots"

In [ ]:
import glob
import hashlib
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import py21cmfast as p21c
from py21cmfast.io.caching import OutputCache, RunCache

warnings.filterwarnings("ignore", category=UserWarning)
assert tuple(int(v) for v in p21c.__version__.split(".")[:2]) >= (4, 3), (
    f"21cmFAST {p21c.__version__}: use 4.3 or later (the 4.2 turnover-mass bug changes the reionization history)")
p21c.config["HALO_CATALOG_MEM_FACTOR"] = 2.0  # as in run_scripts/sim_steps.py
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS.mkdir(exist_ok=True)
print("21cmFAST", p21c.__version__, "| cache:", CACHE_DIR)

# Qin+25 x_HI(z) points, as in plot_all_summaries.ipynb
QIN25_Z = np.array([5.44, 6.35, 6.75, 7.17, 7.7, 8.2, 8.5, 8.85, 9.18, 9.65, 10.01, 10.19, 10.30, 10.87, 11.88, 12.30, 13.144, 14.30])
QIN25_XHI = np.array([0.01, 0.2, 0.3, 0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.84, 0.85, 0.86, 0.9, 0.94, 0.952, 0.96947, 0.9822])

## Functions

In [ ]:
FESC_KEYS = ("F_ESC10_ACG", "ALPHA_ESC", "F_ESC7_MCG")


def make_inputs(fesc):
    """Template inputs in the small box, with optional f_esc overrides."""
    return p21c.InputParameters.from_template(TEMPLATE, random_seed=RANDOM_SEED).evolve_input_structs(
        HII_DIM=HII_DIM, LOWRES_CELL_SIZE_MPC=LOWRES_CELL_SIZE_MPC, N_THREADS=N_THREADS, **fesc)


def fesc_values(inputs):
    return {k: getattr(inputs.astro_params, k) for k in FESC_KEYS}


def history_file(inputs):
    """Saved history of a run, named after a hash of all its inputs."""
    return CACHE_DIR / f"xHI_history_{hashlib.md5(repr(inputs).encode()).hexdigest()[:12]}.json"


def run_history(inputs):
    """Run the box down to the lowest node redshift and return the mean x_HI at every node."""
    cache = OutputCache(CACHE_DIR)
    # Only the lowest node redshift is an output redshift: the run goes down to it (one merger tree rooted there),
    # and 21cmFAST drops each coeval after the next step, so the memory stays flat.
    out_z = [float(min(inputs.node_redshifts))]
    # write=True: every box is written to the cache (the full simulation).
    rows, t0 = [], time.time()
    for coeval, _ in p21c.generate_coeval(inputs=inputs, out_redshifts=out_z, write=True, cache=cache):
        rows.append(dict(z=float(coeval.redshift),
                         xHI=float(np.mean(np.asarray(coeval.ionized_box.neutral_fraction))),
                         Tb=float(np.mean(np.asarray(coeval.brightness_temperature.brightness_temp))),
                         Ts=float(np.mean(np.asarray(coeval.ts_box.spin_temperature)))))
        if len(rows) % 10 == 0 or rows[-1]["z"] <= out_z[0] + 1e-6:
            print(f"z = {rows[-1]['z']:6.3f}   x_HI = {rows[-1]['xHI']:.4f}   {time.time() - t0:6.0f} s")
    return rows


def coevals_cached(inputs):
    """True if every coeval of the run (IonizedBox, TsBox, BrightnessTemp) is in CACHE_DIR."""
    rc = RunCache.from_inputs(inputs, cache=OutputCache(CACHE_DIR))
    return all(Path(p).exists() for d in (rc.IonizedBox, rc.TsBox, rc.BrightnessTemp) for p in d.values())


def get_history(inputs):
    """(z ascending, mean x_HI), from the saved file or from a new run.

    A saved history is used only if the coevals of the run are in the cache too (older versions of
    this notebook cached the ICs, perturbed fields and halo catalogues only); otherwise the run is
    repeated, reusing those.
    """
    fn = history_file(inputs)
    saved = json.load(open(fn))["rows"] if fn.exists() else None
    if saved and not RERUN and "Tb" in saved[0] and coevals_cached(inputs):
        rows = saved
        print("read", fn.name)
    else:
        if saved and not RERUN:
            print(f"{fn.name}: the coevals of this run are not in the cache; running it again")
        rows = run_history(inputs)
        json.dump(dict(fesc=fesc_values(inputs), HII_DIM=HII_DIM, LOWRES_CELL_SIZE_MPC=LOWRES_CELL_SIZE_MPC,
                       N_THREADS=N_THREADS, random_seed=RANDOM_SEED, template=str(TEMPLATE),
                       p21c_version=p21c.__version__, rows=rows), open(fn, "w"), indent=1)
        print("wrote", fn.name)
    z, x = np.array([r["z"] for r in rows]), np.array([r["xHI"] for r in rows])
    o = np.argsort(z)
    return z[o], x[o]


def xhi_from_coeval_logs(pattern):
    """Mean x_HI at each node from the DEBUG argument summaries of the production coeval logs."""
    res = {}
    for fn in sorted(glob.glob(str(pattern))):
        lines = open(fn, errors="replace").read().splitlines()
        for i, line in enumerate(lines):
            if "Calling ComputeBrightnessTemp with following args" in line:
                z = float(lines[i + 1].split()[-1])
                for j in range(i + 2, min(i + 200, len(lines))):
                    if "neutral_fraction:" in lines[j]:
                        vals = re.findall(r"[-+]?\d\.\d+e[-+]\d+", lines[j])
                        if len(vals) == 5:  # 1st, End, Min, Max, Mean
                            res[round(z, 4)] = float(vals[4])
                        break
    z = np.array(sorted(res))
    return z, np.array([res[k] for k in z])


def z_at_xhi(z, x, level):
    """Redshift at which x_HI first drops below level, coming from high redshift (linear interpolation)."""
    o = np.argsort(z)[::-1]
    zz, xx = z[o], x[o]
    below = np.nonzero(xx <= level)[0]
    if len(below) == 0 or below[0] == 0:
        return np.nan
    i = below[0]
    return float(np.interp(level, [xx[i], xx[i - 1]], [zz[i], zz[i - 1]]))


def tau_e(z, x, inputs):
    """Thomson optical depth for a history given at the nodes (x_HI held at its last value below the lowest node)."""
    zz = np.linspace(5.0, z.max(), 400)
    return float(p21c.compute_tau(redshifts=zz, global_xHI=np.interp(zz, z, x), inputs=inputs))

## EOS26_earlyEoR production history

In [ ]:
if not PROD_XHI.exists():
    z, x = xhi_from_coeval_logs(PROD_LOGS)
    np.savetxt(PROD_XHI, np.c_[z, x], fmt=["%9.4f", "%.4e"],
               header=f"EOS26_earlyEoR production: box-mean x_HI at each node, from {PROD_LOGS.relative_to(REPO)}\n redshift   x_HI")
    print("wrote", PROD_XHI.name)
prod_z, prod_x = np.loadtxt(PROD_XHI, unpack=True)
print(f"EOS26_earlyEoR production: {len(prod_z)} nodes, z = {prod_z.min():.3f}-{prod_z.max():.2f}")

## Small-box runs

In [ ]:
histories = {}
for run in RUNS:
    inputs = make_inputs(run["fesc"])
    f = fesc_values(inputs)
    label = run["label"] or "F_ESC10_ACG = {F_ESC10_ACG:.2f}, ALPHA_ESC = {ALPHA_ESC:.2f}, F_ESC7_MCG = {F_ESC7_MCG:.2f}".format(**f)
    print(label)
    z, x = get_history(inputs)
    histories[label] = dict(z=z, x=x, color=run["color"], inputs=inputs)

## Plot

In [ ]:
box = HII_DIM * LOWRES_CELL_SIZE_MPC
fig, ax = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax[0].scatter(QIN25_Z, QIN25_XHI, color="r", zorder=5, label="Qin+25")
lines = [("EOS26_earlyEoR production: F_ESC10 = F_ESC7 = -0.60, ALPHA_ESC = -0.16", prod_z, prod_x, "k")]
lines += [(label, h["z"], h["x"], h["color"]) for label, h in histories.items()]
for label, z, x, c in lines:
    ax[0].plot(z, x, color=c, lw=2, label=label)
    ok = QIN25_Z >= z.min() - 1e-6
    ax[1].plot(QIN25_Z[ok], np.interp(QIN25_Z[ok], z, x) - QIN25_XHI[ok], color=c, lw=2, marker="o", ms=3)
ax[1].axhline(0, color="r")
ax[1].set_ylabel(r"EOS $-$ Qin+25")
ax[1].set_xlabel("Redshift")
ax[0].set_ylabel(r"$\overline{x}_{\rm HI}$")
ax[0].set_xlim(5, 16)
ax[0].legend(fontsize=10, frameon=False, loc="lower right")
ax[0].set_title(f"21cmFAST {p21c.__version__}; small boxes: {box:.0f} cMpc ({HII_DIM}$^3$ at {LOWRES_CELL_SIZE_MPC} cMpc), "
                f"seed {RANDOM_SEED}, N_THREADS {N_THREADS}", fontsize=11)
plt.tight_layout()
fn = PLOTS / f"EOS26_smallbox_xHI_dim{HII_DIM}_cell{LOWRES_CELL_SIZE_MPC}.png"
plt.savefig(fn, dpi=110, bbox_inches="tight")
print("saved", fn.relative_to(REPO))
plt.show()

## Redshifts of xHI levels and tau_e

In [ ]:
levels = [0.9, 0.75, 0.5, 0.25, 0.1, 0.05, 0.01]
ref_inputs = next(iter(histories.values()))["inputs"]
print(f"{'':60s}" + "".join(f"{l:>7}" for l in levels) + "   tau_e")
rows = [("Qin+25", QIN25_Z, QIN25_XHI, None), ("EOS26_earlyEoR production", prod_z, prod_x, ref_inputs)]
rows += [(label, h["z"], h["x"], h["inputs"]) for label, h in histories.items()]
for label, z, x, inp in rows:
    t = f"{tau_e(z, x, inp):8.4f}" if inp is not None else "       -"
    print(f"{label[:60]:60s}" + "".join(f"{z_at_xhi(z, x, l):7.2f}" for l in levels) + t)